# JIN AI — Phase 19: thêm dữ liệu rồi pretrain tiếp (800M token)

Phase 18 đã chứng minh giới hạn nằm ở **dữ liệu**, không ở số step: 300M token nữa
trên đúng corpus 160,9M token (1,86 vòng) đưa val 3,8585 → **3,4839**, nhưng val
chững từ step 8.500 và tăng nhẹ ở 9.155. Kéo dài cosine trên dữ liệu cũ chỉ mua
memorisation.

Notebook này làm hai việc đã nêu:

1. **Thêm dữ liệu** — tải thêm Wikipedia EN từ `wikimedia/wikipedia`
   (`20231101.en`, 41 shard) thay vì dùng lại 160,9M token cũ, và nâng
   `--max-train-tokens` lên **800M**.
2. **Mixture** — trộn thêm `cosmopedia-v2` (SmolLM-Corpus, ODC-BY) ở 15% trọng số,
   để model không chỉ học Wikipedia.

Model lớn hơn thì **không** làm: 124M tham số chỉ có lợi khi đã thấy ≥20B token.

## Cảnh báo cần đọc trước

- **Val set sẽ khác.** Corpus mới có val.bin mới, nên `3,4839` (val cũ) **không so
  trực tiếp** được với số của Phase 19. Notebook vì thế đo **anchor**: chạy
  `best.pt` của Phase 18 trên val.bin **mới** trước khi train, rồi so best mới với
  anchor đó — cùng val, cùng protocol.
- **Cần Internet = On** (tải HF + clone code) và **Accelerator = GPU T4 x2**.
- Thời gian: 800M token / ~10.000 tok/s ≈ **22 giờ**, tức **3 phiên**. Phiên 1 còn
  phải dựng corpus (~2 giờ), nên `--session-max-minutes` được tính từ giờ còn lại
  của phiên chứ không cố định 560.
- Dung lượng `/kaggle/working` là 20 GB. Notebook xoá parquet tải về và
  `curated_v8/documents` ngay sau khi dựng xong bin để không tràn đĩa.


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile

NB_START = time.time()
WORK = "/kaggle/working/repo"
OUT = "/kaggle/working/phase19_output"
os.makedirs(OUT, exist_ok=True)

def _first(pattern, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))
    return hits[0] if hits else None

# ---- code: clone repo mới nhất; Internet off thì dùng bản trong input ----
REPO_URL = "https://github.com/Annguyen0410/JIN-AI-LLM-from-zero-to-hero-ZHIEND.git"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
cloned = False
try:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK],
                   check=True, capture_output=True, timeout=300)
    cloned = os.path.exists(os.path.join(WORK, "train_gpt2.py"))
except Exception as exc:
    print("clone không chạy được (Internet off?) -> dùng bản trong dataset:", exc)

if not cloned:
    entry = _first("train_gpt2.py")
    zipped = _first("*.zip")
    if entry:
        shutil.copytree(os.path.dirname(entry), WORK, ignore=shutil.ignore_patterns(
            ".git", "__pycache__", "*.pyc", "data", "checkpoints", "cache"))
    elif zipped:
        os.makedirs(WORK, exist_ok=True)
        zipfile.ZipFile(zipped).extractall(WORK)
        inner = [d for d in os.listdir(WORK)
                 if os.path.isfile(os.path.join(WORK, d, "train_gpt2.py"))]
        if inner:
            WORK = os.path.join(WORK, inner[0])
    print("code: dùng bản trong input (có thể cũ hơn)")

os.chdir(WORK)
CODE_REV = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                          text=True).stdout.strip() or "unknown"
print("code revision:", CODE_REV)

# ---- điểm khởi đầu: best.pt của Phase 18 (300M token, val 3,4839) ----
PREV = (_first("run_400m/best.pt")
        or _first("phase18_output/**/best.pt")
        or ("/kaggle/working/phase18_output/run_400m/best.pt"
            if os.path.exists("/kaggle/working/phase18_output/run_400m/best.pt") else None))
if not PREV:
    raise SystemExit("KHÔNG thấy run_400m/best.pt của Phase 18 — Add Input output của "
                     "version đã train 300M (Stage G) rồi chạy lại cell này.")

PREV_MAN = PREV + ".manifest.json"
if os.path.exists(PREV_MAN):
    man = json.load(open(PREV_MAN, encoding="utf-8"))
    print("prev:", {k: man.get(k) for k in
                    ("iteration", "best_val_loss", "additional_tokens_this_run",
                     "parent_sha256", "init_mode")})
    assert man.get("additional_tokens_this_run", 0) > 250_000_000, \
        "best.pt này chưa train 300M token — không phải output Stage G của Phase 18"
    assert man.get("best_val_loss", 99) < 3.7, "best.pt Phase 18 phải có val < 3,7"

TARGET_TOKENS = 800_000_000          # đổi thành 1_000_000_000 nếu muốn đi tiếp 1B
TOKENS_PER_STEP = 4 * 8 * 1024
TOTAL_STEPS = TARGET_TOKENS // TOKENS_PER_STEP
CHOSEN_LR = "1e-4"                   # giữ recipe Phase 17/18: chỉ dữ liệu thay đổi
SESSION_CAP_MIN = 11 * 60             # trần 1 phiên Kaggle 12h, chừa 1h cho build/lưu

CONFIG = {
    "prev_checkpoint": PREV,
    "reuse_corpus": _first("gpt2_corpus_v8/manifest.json"),
    "out_dir": OUT,
    "target_tokens": TARGET_TOKENS,
    "total_steps": TOTAL_STEPS,
    "code_revision": CODE_REV,
}
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("checkpoint:", round(os.path.getsize(PREV) / 1e9, 2), "GB")
print(f"budget: {TOTAL_STEPS:,} step x {TOKENS_PER_STEP:,} = "
      f"{TOTAL_STEPS * TOKENS_PER_STEP / 1e9:.2f}B token")
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)


## Stage 1. Dữ liệu: tải thêm Wikipedia + cosmopedia, dựng corpus v8

Nguồn (đều đã có entry review trong `scripts/license_policy.py`, nên
`--license-policy enforce` kiểm được giấy phép chứ không tin tên folder):

| nguồn | lấy từ | license đã review | trọng số trong `--mix` |
|---|---|---|---|
| `20231101.en` (Wikipedia EN) | `wikimedia/wikipedia` | `wikipedia-en-20231101` (CC-BY-SA-3.0 + GFDL) | 0,85 |
| `cosmopedia-v2` | `HuggingFaceTB/smollm-corpus` | `cosmopedia-v2-sample` (ODC-BY 1.0, text tổng hợp) | 0,15 |

Chỉ cần ~5 shard Wikipedia (~780k bài ≈ 740M token) + 1 shard cosmopedia
(~376k bài). `build_tokenized_corpus.py` cắt **riêng từng nguồn** về đúng tỷ lệ
của `--mix` (15% cho cosmopedia) nên kết quả không phụ thuộc thứ tự file; cell in
lại `token_composition` và sẽ dừng nếu nguồn thứ hai bị rơi mất. Nếu output của
phiên trước đã có `gpt2_corpus_v8`, cell bỏ qua toàn bộ bước tải/dựng.


In [ ]:
LICENSE = {
    "20231101.en": {
        "license": "CC-BY-SA-3.0 and GFDL (dataset card)",
        "source_url": "https://huggingface.co/datasets/wikimedia/wikipedia",
        "policy_source": "wikipedia-en-20231101",
        "revision": "main (file sha256 recorded below)",
        "text_columns": ["text"], "language": "en",
    },
    "cosmopedia-v2": {
        "license": "ODC-BY 1.0 (dataset card)",
        "source_url": "https://huggingface.co/datasets/HuggingFaceTB/smollm-corpus",
        "policy_source": "cosmopedia-v2-sample",
        "revision": "main (file sha256 recorded below)",
        "text_columns": ["text"], "language": "en",
    },
}
WIKI_REPO, WIKI_CONFIG, WIKI_TOTAL = "wikimedia/wikipedia", "20231101.en", 41
COSMO_REPO, COSMO_CONFIG, COSMO_TOTAL = "HuggingFaceTB/smollm-corpus", "cosmopedia-v2", 104
TOKENS_PER_DOC = 950          # đo từ corpus v7: 160,9M token / ~170k bài
WIKI_SHARE, COSMO_SHARE = 0.85, 0.15
MAX_DOCS = 1_000_000          # cap ingest cho MỖI source (wiki cần ~716k bài)
STAGE = "/kaggle/working/phase19_inputs"
CURATED = "data/curated_v8"
TOKENS = "data/gpt2_corpus_v8"

def _sha256(path, block=1 << 22):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

def _rows(path):
    import pyarrow.parquet as pq
    return pq.ParquetFile(path).metadata.num_rows

if CONFIG["reuse_corpus"]:
    shutil.rmtree(TOKENS, ignore_errors=True)
    shutil.copytree(os.path.dirname(CONFIG["reuse_corpus"]), TOKENS)
    print("corpus: dùng lại từ input ->", TOKENS)
else:
    from huggingface_hub import hf_hub_download

    shutil.rmtree(STAGE, ignore_errors=True)
    os.makedirs(STAGE, exist_ok=True)
    want_docs = int(TARGET_TOKENS * WIKI_SHARE / TOKENS_PER_DOC * 1.02)
    print(f"cần ~{want_docs:,} bài Wikipedia cho {TARGET_TOKENS * WIKI_SHARE / 1e6:.0f}M token")

    downloaded = []
    wiki_rows = 0
    for i in range(WIKI_TOTAL):
        if wiki_rows >= want_docs:
            break
        name = f"{WIKI_CONFIG}/train-{i:05d}-of-{WIKI_TOTAL:05d}.parquet"
        path = hf_hub_download(WIKI_REPO, name, repo_type="dataset",
                               local_dir=os.path.join(STAGE, "wikipedia"))
        wiki_rows += _rows(path)
        downloaded.append(path)
        print(f"  wiki shard {i}: +{_rows(path):,} bài -> {wiki_rows:,} "
              f"({os.path.getsize(path) / 1e6:.0f} MB)")
    if wiki_rows < want_docs * 0.98:
        raise SystemExit(f"chỉ tải được {wiki_rows:,} bài (< {want_docs:,}). "
                         "Tăng số shard hoặc bật Internet rồi chạy lại.")

    cosmo = []
    for i in range(COSMO_TOTAL):
        name = f"{COSMO_CONFIG}/train-{i:05d}-of-{COSMO_TOTAL:05d}.parquet"
        cosmo.append(hf_hub_download(COSMO_REPO, name, repo_type="dataset",
                                     local_dir=os.path.join(STAGE, "cosmopedia")))
        break        # 1 shard ~376k bài ≈ 490M token, quá đủ cho 120M token ở 15%
    print(f"  cosmopedia: +{_rows(cosmo[0]):,} bài "
          f"({os.path.getsize(cosmo[0]) / 1e6:.0f} MB)")

    provenance = {"downloaded_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
                  "files": [{"path": os.path.basename(p), "bytes": os.path.getsize(p),
                             "sha256": _sha256(p), "rows": _rows(p)}
                            for p in downloaded + cosmo]}
    json.dump(provenance, open("/kaggle/working/phase19_downloads.json", "w",
                              encoding="utf-8"), indent=2)
    print("download provenance -> /kaggle/working/phase19_downloads.json")

    json.dump(LICENSE, open("/kaggle/working/license_map_v8.json", "w",
                            encoding="utf-8"), indent=2)
    # Từng nguồn một (KHÔNG --all): --all kéo theo mọi nguồn nó tự thấy, kể cả
    # prepared_v4/seed và bất kỳ dataset nào khác đang attach, làm corpus và
    # thời gian dedup phình ra ngoài ý muốn.
    for src_id in (WIKI_CONFIG, COSMO_CONFIG.replace("-", "_")):
        subprocess.run([sys.executable, "scripts/ingest_corpus.py",
                        "--input-root", STAGE, "--source", src_id,
                        "--license-map", "/kaggle/working/license_map_v8.json",
                        "--max-docs-per-source", str(MAX_DOCS),
                        "--out-root", CURATED]).check_returncode()

    # parquet đã ingest xong -> xoá để lấy lại ~2,6 GB
    for path in downloaded + cosmo:
        os.remove(path)

    subprocess.run([sys.executable, "scripts/verify_corpus.py",
                    "--in-root", CURATED, "--out-root", CURATED,
                    "--pii-policy", "exclude", "--license-policy", "enforce"]).check_returncode()

    MIX = f"{WIKI_CONFIG}:{WIKI_SHARE},{COSMO_CONFIG.replace('-', '_')}:{COSMO_SHARE}"
    subprocess.run([sys.executable, "scripts/build_tokenized_corpus.py",
                    "--splits", f"{CURATED}/splits", "--out", TOKENS,
                    "--max-train-tokens", str(TARGET_TOKENS), "--mix", MIX]).check_returncode()

    # documents/ chỉ dùng cho verify; splits đã ghi xong -> xoá để không tràn đĩa
    shutil.rmtree(f"{CURATED}/documents", ignore_errors=True)

tok_man = json.load(open(f"{TOKENS}/manifest.json", encoding="utf-8"))
print(json.dumps({k: tok_man.get(k) for k in ("train", "val")}, indent=2)[:1200])
TRAIN_DATA = TOKENS
n_train = tok_man["train"].get("tokens") or tok_man["train"].get("num_tokens")
if not n_train:
    raise SystemExit("manifest không có số token train")
print(f"train bin: {n_train:,} token = {n_train / TOKENS_PER_STEP:,.0f} step")
if n_train < TARGET_TOKENS * 0.9:
    raise SystemExit(f"chỉ dựng được {n_train:,} token (<90% của {TARGET_TOKENS:,}). "
                     "Thêm shard hoặc giảm TARGET_TOKENS, đừng train lặp dữ liệu.")

comp = tok_man["train"].get("token_composition") or {}
cosmo_id = COSMO_CONFIG.replace("-", "_")
if cosmo_id not in comp:
    raise SystemExit(f"nguồn thứ hai rơi mất khỏi corpus: {comp}. Mixture là mục tiêu "
                     "của phase này, nên dừng thay vì train wiki-only.")
share = comp[cosmo_id] / n_train
print(f"mixture: cosmopedia chiếm {share:.1%} token train (mục tiêu {COSMO_SHARE:.0%})")
if not 0.10 <= share <= 0.20:
    raise SystemExit(f"tỷ lệ cosmopedia {share:.1%} lệch khỏi 15% — kiểm tra lại --mix")
print(f"corpus {n_train:,} token -> {TARGET_TOKENS / n_train:.2f} vòng "
      f"(Phase 17+18 đã dùng 400M token = {400_000_000 / n_train:.2f} vòng trên dữ liệu cũ)")
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)


## Stage 2. Anchor: chấm `best.pt` Phase 18 trên val.bin **mới**

Đây là mốc so sánh của Phase 19. Cùng val.bin, cùng seed (24), cùng batch 2, cùng
20 iteration, cùng `train_gpt2.evaluate` mà trainer dùng → số anchor và số cuối kỳ
cùng thang đo. Cell cũng là smoke-test load checkpoint: val ≈ 3,5 chứ không phải
≈ 10,8 của random init.


In [ ]:
import torch, train_gpt2 as T

def eval_on_val_bin(ckpt_path, data_dir, iters=20, seq_len=1024, label=""):
    """Val loss bằng đúng hàm/seed/batch mà trainer dùng (không viết lại eval)."""
    tok = T.GPT2BPETokenizer()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = T.build_model("gpt2_small_reference", tok.vocab_size, seq_len, device,
                          False, 0.0)
    lineage = T.init_from_checkpoint(model, tok, ckpt_path, log_fn=lambda m: None)
    batcher = T.ShardBatcher(f"{data_dir}/val.bin", seq_len, 2, device, 24, tok.eot_id)
    res = T.evaluate(model, batcher, iters, torch.float16, device == "cuda")
    print(f"{label} val_loss {res['loss']} | ppl {res['perplexity']} | "
          f"tokens_scored {res['tokens_scored']} | {device}")
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return {"checkpoint": ckpt_path, "label": label, "val_loss": res["loss"],
            "val_ppl": res["perplexity"], "tokens_scored": res["tokens_scored"],
            "iters": iters, "seq_len": seq_len, "val_bin": f"{data_dir}/val.bin",
            "val_bin_sha256": tok_man["val"]["sha256"],
            "init_mode": lineage["init_mode"], "parent_sha256": lineage["parent_sha256"]}

ANCHOR = eval_on_val_bin(CONFIG["prev_checkpoint"], TRAIN_DATA,
                         label="anchor (Phase 18 best @ val v8)")
assert ANCHOR["val_loss"] < 6.0, "checkpoint Phase 18 không load đúng trên val mới"
json.dump(ANCHOR, open(f"{OUT}/anchor_phase18_on_v8val.json", "w", encoding="utf-8"),
          indent=2)


## Stage 3. Smoke 10 step — đúng lệnh của Stage G

Không đo chất lượng; chỉ để chắc `--init-from` + corpus mới + save/manifest chạy
được, trước khi đốt 3 phiên GPU.


In [ ]:
smoke_dir = f"{OUT}/smoke"
shutil.rmtree(smoke_dir, ignore_errors=True)
subprocess.run([sys.executable, "train_gpt2.py",
                "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                "--init-from", CONFIG["prev_checkpoint"], "--continue-pretrain",
                "--out-dir", smoke_dir, "--seq-len", "1024",
                "--batch-size", "4", "--grad-accum", "8",
                "--max-steps", "10", "--warmup", "2", "--lr", "1e-4",
                "--eval-interval", "5", "--eval-iters", "2",
                "--save-interval", "10", "--seed", "17"]).check_returncode()

smoke_man = json.load(open(f"{smoke_dir}/latest.pt.manifest.json", encoding="utf-8"))
prev_man = json.load(open(PREV_MAN, encoding="utf-8")) if os.path.exists(PREV_MAN) else {}
print("smoke val:", smoke_man["best_val_loss"], "| init_mode:", smoke_man.get("init_mode"),
      "| parent:", str(smoke_man.get("parent_sha256"))[:12])
assert smoke_man.get("init_mode") == "continued_pretrain", smoke_man.get("init_mode")
if prev_man.get("sha256"):
    assert smoke_man.get("parent_sha256") == prev_man["sha256"], \
        "smoke không init từ đúng best.pt của Phase 18"
for p in glob.glob(f"{smoke_dir}/*.pt"):
    os.remove(p)


## G. Train tiếp — 800M token (24.414 step)

Recipe giữ nguyên như Phase 17/18 (cosine 1e-4 → 1e-5, warmup 300, eval 500×20) để
thay đổi duy nhất là **dữ liệu**. `--init-from` = experiment mới: optimizer,
schedule, `best_val` riêng, manifest ghi `parent_sha256`.

`--session-max-minutes` tính từ giờ còn lại của phiên (trần 11 giờ) chứ không cố
định 560, vì phiên 1 còn dựng corpus: hết giờ thì nó lưu rồi thoát, `total_steps`
không đổi.


In [ ]:
elapsed_min = (time.time() - NB_START) / 60.0
SESSION_MIN = int(max(90, SESSION_CAP_MIN - elapsed_min))
print(f"{TOTAL_STEPS:,} step x {TOKENS_PER_STEP:,} = "
      f"{TOTAL_STEPS * TOKENS_PER_STEP / 1e9:.2f}B token")
print(f"phiên này đã dùng {elapsed_min:.0f} phút -> session budget {SESSION_MIN} phút")
if elapsed_min > SESSION_CAP_MIN - 60:
    raise SystemExit("Không còn đủ giờ cho một phiên train có nghĩa. Save Version, "
                     "mở phiên mới, chạy Stage 1 rồi cell G2.")

subprocess.run([sys.executable, "train_gpt2.py",
                "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                "--init-from", CONFIG["prev_checkpoint"], "--continue-pretrain",
                "--out-dir", f"{OUT}/run_800m", "--seq-len", "1024",
                "--batch-size", "4", "--grad-accum", "8",
                "--max-steps", str(TOTAL_STEPS), "--warmup", "300", "--lr", CHOSEN_LR,
                "--eval-interval", "500", "--eval-iters", "20",
                "--save-interval", "250", "--session-max-minutes", str(SESSION_MIN),
                "--seed", "17"]).check_returncode()


## G2. Phiên sau — resume đúng chỗ đã dừng

Save Version → phiên mới → Add Input output của version vừa train → chạy Stage 1
(clone + corpus reuse) rồi **chỉ cell này**. Không truyền `--init-from`, chỉ
`--resume-from`; không thấy checkpoint thì nó in cảnh báo chứ không train lại từ đầu.

Điều kiện: `code_revision` phải giống phiên trước (fingerprint chứa nó). Notebook
clone `main`, nên **đừng sửa `train_gpt2.py`** giữa các phiên.


In [ ]:
PREV_RUN = (_first("run_800m/latest.pt")
            or ("/kaggle/working/phase19_output/run_800m/latest.pt"
                if os.path.exists("/kaggle/working/phase19_output/run_800m/latest.pt") else None))
if PREV_RUN and os.path.exists(PREV_RUN):
    man = json.load(open(PREV_RUN + ".manifest.json", encoding="utf-8"))
    print("resume từ step", man.get("iteration"), "| best val", man.get("best_val_loss"))
    elapsed_min = (time.time() - NB_START) / 60.0
    SESSION_MIN = int(max(90, SESSION_CAP_MIN - elapsed_min))
    print(f"session budget {SESSION_MIN} phút")
    subprocess.run([sys.executable, "train_gpt2.py",
                    "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                    "--resume-from", PREV_RUN, "--out-dir", f"{OUT}/run_800m",
                    "--seq-len", "1024", "--batch-size", "4", "--grad-accum", "8",
                    "--max-steps", str(TOTAL_STEPS), "--warmup", "300", "--lr", CHOSEN_LR,
                    "--eval-interval", "500", "--eval-iters", "20",
                    "--save-interval", "250", "--session-max-minutes", str(SESSION_MIN),
                    "--seed", "17"]).check_returncode()
else:
    print("Không có latest.pt của run_800m — chưa chạy Stage G hoặc chưa attach output.")


## H. Báo cáo — so với **anchor**, không so với số của val cũ

Kiểm SHA khớp manifest, chấm `best.pt` mới trên đúng val.bin vừa dùng cho anchor,
rồi in chênh lệch. `3,4839` của Phase 18 là val.bin **khác**, nên chỉ ghi lại để
tham khảo, không trừ vào số mới.


In [ ]:
import hashlib

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()

run_dir = f"{OUT}/run_800m"
report = {"phase": 19, "val_bin": ANCHOR["val_bin"], "val_bin_sha256": ANCHOR["val_bin_sha256"],
          "anchor": ANCHOR, "checkpoints": {}, "not_comparable": {
              "phase18_best_val_loss_on_its_own_val": 3.4839,
              "why": "Phase 19 dùng val.bin mới của corpus v8; số Phase 18 đo trên val v7"}}
for name in ("latest.pt", "best.pt"):
    path = f"{run_dir}/{name}"
    if not os.path.exists(path):
        continue
    man = json.load(open(path + ".manifest.json", encoding="utf-8"))
    assert sha(path) == man["sha256"], f"{name}: hash không khớp manifest"
    report["checkpoints"][name] = {k: man.get(k) for k in
                                   ("sha256", "iteration", "best_val_loss",
                                    "additional_tokens_this_run", "init_mode",
                                    "parent_sha256", "tokens_seen")}
    print(f"{name}: sha256={man['sha256'][:16]}… step={man['iteration']} "
          f"val={man['best_val_loss']:.4f} tokens={man['additional_tokens_this_run']:,} "
          f"parent={man['parent_sha256'][:12]}… mode={man['init_mode']}")

if os.path.exists(f"{run_dir}/best.pt"):
    FINAL = eval_on_val_bin(f"{run_dir}/best.pt", TRAIN_DATA,
                            label="Phase 19 best @ val v8")
    report["final"] = FINAL
    delta = ANCHOR["val_loss"] - FINAL["val_loss"]
    report["delta_vs_anchor"] = round(delta, 4)
    print(f"\nbest mới {FINAL['val_loss']:.4f} vs anchor {ANCHOR['val_loss']:.4f} "
          f"-> {'giảm' if delta > 0 else 'tăng'} {abs(delta):.4f} "
          f"(cùng val.bin {ANCHOR['val_bin_sha256'][:12]}…)")
else:
    print("chưa có best.pt của run_800m")

json.dump(report, open("/kaggle/working/phase19_report.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False)
print("report -> /kaggle/working/phase19_report.json")


## Đọc kết quả này cho đúng

- Số duy nhất so được với Phase 18 là **ppl trên val.bin v7**; Phase 19 đo trên
  **val.bin v8**, nên phép so có nghĩa là *best mới vs anchor* trong cùng notebook.
- Corpus v8 trộn 15% văn bản tổng hợp (cosmopedia) và vẫn 100% tiếng Anh. Mọi phát
  biểu về tiếng Việt vẫn là **chưa đo**.
- Đây vẫn là **perplexity**, không phải năng lực. Benchmark 73 task phải chạy riêng
  (`scripts/gpt2_bench.py`) trên checkpoint mới; số cuối vẫn là 1/73 của Phase 17.
- Nếu val tiếp tục giảm tới step cuối (24.414) thì bước sau là thêm dữ liệu nữa
  (Wikipedia full 6,4M bài ≈ 5–6B token), không phải tăng step trên corpus v8.
